# Reproducible Meta‑Analysis of Autonomous Robotic Imaging in Agriculture

## Overview
This notebook contains a complete, reproducible Python workflow used in the systematic review and meta‑analysis of autonomous robotic systems for agricultural image data collection across plant and livestock domains.

## Data
- Curated dataset of **155 full-text studies assessed**, with **43 contributing to quantitative synthesis**
- Summary of **named publicly available datasets**, including access status and retrieval dates
- Data sources and variable definitions are documented in the accompanying repository

## Methods
All analyses were implemented from first principles in Python using inverse‑variance–weighted random‑effects models on the logit‑transformed F1 scale, with back‑transformation for interpretable reporting.

## Reproducibility
This notebook is released in support of open and transparent research. All code, data files, and outputs required to reproduce the reported results are publicly available in the associated GitHub repository and archived on Zenodo.

---

In [ ]:
# Imports necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler

In [ ]:
def logit(p):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return np.log(p / (1 - p))

def invlogit(x):
    return np.exp(x) / (1 + np.exp(x))

def _pct_formatter(x, pos):
    return f"{x*100:.0f}%"

In [ ]:
# data loading
df = pd.read_csv('masterfile_meta.csv')
df.head()

In [ ]:
# Standardize column names (strip spaces, unify)
df.columns = [c.strip() for c in df.columns]

# Convenience renames (typing-safe because our header has parentheses/spaces)
rename_map = {
    'Agricultural_Domain': 'Domain',
    'Other_sensors': 'Other_Sensors',
    'Resolution (px)': 'Resolution_px',
    'Resolution (mp)': 'Resolution_MP',
    'Data_Volume(GB)': 'Data_Volume_GB',
    'Labelled_files_gen': 'N_Labelled',
    'Raw_images_Collected':'Images_Collected'
}
df = df.rename(columns=rename_map)

# Normalize autonomy strings (fix typos, case)
def norm_autonomy(x):
    if pd.isna(x): return np.nan
    x = str(x).strip().lower()
    x = x.replace('autonomus', 'autonomous')  # fix typo
    if 'fully' in x: return 'fully autonomous'
    if 'semi' in x:  return 'semi autonomous'
    if 'manual' in x: return 'manual'
    return x

for c in ['Autonomy_Robot','Autonomy_Imaging','Labeling_Method']:
    if c in df.columns:
        df[c] = df[c].apply(norm_autonomy)

# Ensure numeric metrics (coerce errors)
metric_cols = ['Accuracy_%','Precision_%','Recall_%','F1_score_%','N_Labelled','Raw_images_Collected','Resolution_MP', 'Data_Volume_GB']
for m in metric_cols:
    if m in df.columns:
        df[m] = pd.to_numeric(df[m], errors='coerce')

print("Loaded rows:", len(df))
df.head(10)

### Temporal robotic distributions

In [ ]:
# KEEP ONLY REQUIRED COLUMNS
plot_df = df[['Publication_Year', 'Robot_Type']].copy()

# COUNT STUDIES PER YEAR AND ROBOT TYPE
plot_df = (
    plot_df
    .groupby(['Publication_Year', 'Robot_Type'])
    .size()
    .reset_index(name='Studies')
)

# ENSURE ALL YEARS APPEAR
all_years = np.arange(
    plot_df['Publication_Year'].min(),
    plot_df['Publication_Year'].max() + 1
)

robot_types = ['UGV', 'UAV', 'Hybrid']

# Create all combinations
full_index = pd.MultiIndex.from_product(
    [all_years, robot_types],
    names=['Publication_Year', 'Robot_Type']
)

plot_df = (
    plot_df
    .set_index(['Publication_Year', 'Robot_Type'])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

# MAP ROBOT TYPES TO Y POSITIONS
y_map = {
    'UGV': 1,
    'UAV': 2,
    'Hybrid': 3
}

plot_df['y'] = plot_df['Robot_Type'].map(y_map)

# COLOR PALETTE
colors = {
    'UGV': '#4C78A8',
    'UAV': '#F58518',
    'Hybrid': '#54A24B'
}

# FIGURE STYLE
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "font.size": 10,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

# CREATE FIGURE
fig, ax = plt.subplots(figsize=(8.7, 5.1))

# PLOT BUBBLES
for robot in robot_types:
    subset = plot_df[plot_df['Robot_Type'] == robot]
    ax.scatter(
        subset['Publication_Year'],
        subset['y'],
        s=subset['Studies'] * 220,   # bubble scaling
        color=colors[robot],
        alpha=0.80,
        edgecolors='black',
        linewidth=0.7,
        label=robot
    )

# ADD LABELS INSIDE BUBBLES
for _, row in plot_df.iterrows():
    if row['Studies'] > 0:
        ax.text(
            row['Publication_Year'],
            row['y'],
            str(int(row['Studies'])),
            ha='center',
            va='center',
            fontsize=9,
            fontweight='bold',
            color='white'
        )

# AXES
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(['UGV', 'UAV', 'Hybrid'])

ax.set_xlabel('Publication Year')
ax.set_ylabel('Robot Type')

# SHOW ALL YEARS
ax.set_xticks(all_years)

ax.set_xlim(all_years.min() - 0.5,
            all_years.max() + 0.5)

# GRID & SPINES
ax.grid(axis='x',
        linestyle='--',
        alpha=0.25)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# LEGEND
ax.legend(
    title='Robot Type',
    frameon=False,
    loc='upper left',
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.savefig(
    "bubble_timeline_robotics.svg",
    bbox_inches='tight'
)

plt.show()

### Geographic patterns and agri-domains

In [ ]:
# Create cross-tabulation
country_domain = pd.crosstab(df['Country'], df['Domain'])

# Plot heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(country_domain, annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': 'Number of Studies'}, ax=ax)
ax.set_xlabel('Agricultural Domain', fontsize=12, fontweight='bold')
ax.set_ylabel('Country', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('Heatmap wordcloud_species20120.svg')
plt.show()

### Targeted species

In [ ]:
from wordcloud import WordCloud

# Separate plant and livestock
plant_df = df[df['Domain'] == 'Plant']['Specific_Target'].dropna()
livestock_df = df[df['Domain'] == 'Livestock']['Specific_Target'].dropna()

# Join all species into single strings
plant_text = ' '.join(plant_df.astype(str).tolist())
livestock_text = ' '.join(livestock_df.astype(str).tolist())

# Color function for plant side — green/purple palette
def plant_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    colors = ['#2e7d32', '#6a0dad', '#1565c0', '#f9a825', '#00695c', '#4527a0']
    import random
    return random.choice(colors)

# Color function for livestock side — orange/pink/dark palette
def livestock_color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    colors = ['#e65100', '#c2185b', '#4a148c', '#1a237e', '#212121', '#f06292']
    import random
    return random.choice(colors)

# Generate wordclouds
wc_plant = WordCloud(
    width=800, height=600,
    background_color='white',
    max_words=200,
    collocations=False,
    prefer_horizontal=0.7
).generate(plant_text)

wc_livestock = WordCloud(
    width=800, height=600,
    background_color='white',
    max_words=200,
    collocations=False,
    prefer_horizontal=0.7
).generate(livestock_text)

# Apply color functions
wc_plant.recolor(color_func=plant_color_func)
wc_livestock.recolor(color_func=livestock_color_func)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

axes[0].imshow(wc_plant, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('(A)', fontsize=13, fontweight='bold', loc='left', pad=10)

axes[1].imshow(wc_livestock, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('(B)', fontsize=13, fontweight='bold', loc='left', pad=10)

plt.tight_layout()
plt.savefig('wordcloud_species.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
for value, count in df['Specific_Target'].value_counts().items():
    print(f"{value}: {count}")

### Sensing modality trend over time

In [ ]:
def sensor_evolution(df):
    # Clean and prepare sensor data
    df_clean = df.dropna(subset=['Imaging_Type', 'Publication_Year']).copy()
    
    # Count sensors per study
    df_clean['Sensor_Count_Estimate'] = df_clean['Number_of_Sensors'].fillna(1)
    
    # Sensor type evolution
    sensor_time = pd.crosstab(df_clean['Publication_Year'], df_clean['Imaging_Type'])
    
    plt.figure(figsize=(12, 7))
    
    plt.subplot(2, 1, 1)
    sensor_time.plot(kind='bar', stacked=True, ax=plt.gca(), 
                    colormap='plasma', width=0.8)
    plt.title('-A-', fontsize=14, fontweight='bold')
    plt.xlabel('Publication Year')
    plt.ylabel('Number of Studies')
    #plt.legend(title='Sensor Type', bbox_to_anchor=(1.05, 1), loc='upper right')
    plt.legend(title='Sensor Type', loc='upper left')

    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    # Average number of sensors over time
    sensors_per_year = df_clean.groupby('Publication_Year')['Sensor_Count_Estimate'].mean()
    studies_per_year = df_clean.groupby('Publication_Year').size()
    
    ax1 = plt.gca()
    ax2 = ax1.twinx()
    
    line1 = ax1.plot(sensors_per_year.index, sensors_per_year.values, 
                    'o-', color='red', linewidth=2, markersize=8, label='Avg Sensors/Study')
    ax1.set_ylabel('Average Number of Sensors', color='red')
    ax1.tick_params(axis='y', labelcolor='red')
    ax1.set_ylim(bottom=0)
    
    line2 = ax2.bar(studies_per_year.index, studies_per_year.values, 
                   alpha=0.3, color='blue', label='Number of Studies')
    ax2.set_ylabel('Number of Studies', color='blue')
    ax2.tick_params(axis='y', labelcolor='blue')
    
    lines = line1 + [line2]
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc='upper left')
    
    plt.title('-B-', fontsize=14, fontweight='bold')
    plt.xlabel('Publication Year')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('sensors.png', dpi=600, bbox_inches='tight')
    plt.show()
    
    # Print trends
    print("Sensor Type Popularity Over Time:")
    for sensor in sensor_time.columns:
        trend = sensor_time[sensor]
        if trend.sum() > 0:  # Only print sensors that appear
            max_year = trend.idxmax()
            max_count = trend.max()
            print(f"{sensor}: Peak in {max_year} ({max_count} studies)")
    
    return sensor_time, sensors_per_year

sensor_evolution, sensor_complexity = sensor_evolution(df)

In [ ]:
for value, count in df['Imaging_Type'].value_counts().items():
    percentage = (count / len(df)) * 100
    print(f"{value}: {count} ({percentage:.1f}%)")

### Datasets and data availability

In [ ]:
def dataset_accessibility(df):
    availability_counts = df['Dataset_Availability'].value_counts()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))
    
    # Define consistent color mapping
    color_map = {
        'Public': '#2ecc71',      # Green
        'Private': '#e74c3c',     # Red
        'Upon Request': '#f39c12'  # Orange
    }
    
    # Get colors in the order of availability_counts
    colors = [color_map[category] for category in availability_counts.index]
    
    wedges, texts, autotexts = ax1.pie(availability_counts.values, 
                                       labels=availability_counts.index,
                                       autopct='%1.1f%%',
                                       colors=colors,
                                       startangle=90,
                                       pctdistance=0.85)
    
    centre_circle = plt.Circle((0,0), 0.70, fc='white')
    ax1.add_artist(centre_circle)
    
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')
    
    # ax1.set_title('Dataset Availability Distribution', fontsize=12, fontweight='bold')

    if 'Publication_Year' in df.columns:
        availability_trend = pd.crosstab(df['Publication_Year'], df['Dataset_Availability'], normalize='index') * 100
        
        # Plot with consistent colors using the color_map
        for category in availability_trend.columns:
            ax2.plot(availability_trend.index, availability_trend[category], 
                    marker='o', label=category, color=color_map[category], linewidth=2, markersize=6)
        
        ax2.set_xlabel('Publication Year', fontsize=11)
        ax2.set_ylabel('Percentage (%)', fontsize=11)
        # ax2.set_title('Dataset Availability Trend Over Time', fontsize=12, fontweight='bold')
        ax2.legend(title='Availability', loc='best')
        ax2.grid(True, alpha=0.3, linestyle='--')
    else:
        ax2.text(0.5, 0.5, 'Publication_Year column not found',
                 ha='center', va='center', transform=ax2.transAxes, fontsize=12)
        ax2.set_title('Trend Analysis Unavailable', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('dataset_availability.png', dpi=600, bbox_inches='tight')
    plt.show()
    
    # Calculate scores (fixed the duplicate print statement)
    total = availability_counts.sum()
    public_score = (availability_counts.get('Public', 0) / total) * 100
    private_score = (availability_counts.get('Private', 0) / total) * 100
    upon_request_score = (availability_counts.get('Upon Request', 0) / total) * 100
    total_restricted = private_score + upon_request_score
    
    print("Dataset Accessibility Analysis:")
    print(f"Publicly available datasets: {public_score:.1f}%")
    print(f"Privately available datasets: {private_score:.1f}%")  # Fixed: was duplicate 'Publicly'
    print(f"Available upon request: {upon_request_score:.1f}%")
    print(f"Total restricted datasets: {total_restricted:.1f}%")
    
    return availability_counts

availability_data = dataset_accessibility(df)

# STATISTICAL ANALYSIS

In [ ]:
PRIMARY_METRIC = 'F1_score_%'

if PRIMARY_METRIC not in df.columns:
    raise ValueError(f"PRIMARY_METRIC '{PRIMARY_METRIC}' not in columns. Available: {sorted(metric_cols)}")

# Convert % to proportion [0,1]
vals = df[PRIMARY_METRIC] / (100.0 if PRIMARY_METRIC.endswith('%') else 1.0)

# Clip to avoid infinite logits
eps = 1e-6
p = vals.clip(eps, 1 - eps)
df['_p'] = p

# Logit transform and SE via delta method for proportions
# Var(logit(p)) ≈ 1 / (n * p * (1-p)); SE = sqrt(Var)
df['_n'] = df['N_eval']
df['_vi'] = 1.0 / (df['_n'] * df['_p'] * (1 - df['_p']))
df['_sei'] = np.sqrt(df['_vi'])
df['_yi'] = np.log(df['_p'] / (1 - df['_p']))  # logit(p)

# Keep studies with defined variance
meta_df = df.loc[df['Robot_Type'].notna() & df['_vi'].notna() & np.isfinite(df['_vi']) & np.isfinite(df['_yi'])].copy()
meta_df = meta_df.reset_index(drop=True)

print(f"Primary metric: {PRIMARY_METRIC}")
print(f"Meta-analytic masterfile: {len(meta_df)}")
meta_df[['Study_ID','Domain','Robot_Type','Imaging_Type','AI_Driven_Technology',
       'Model/Network','N_eval',PRIMARY_METRIC,'_p','_yi','_vi','_sei']].head(50)

### Random-effects (DerSimonian–Laird), heterogeneity & back-transform pooled effect to %

In [ ]:
yi = meta_df['_yi'].values
vi = meta_df['_vi'].values
wi = 1/vi

# Fixed-effect pooled logit
mu_fixed = np.sum(wi*yi) / np.sum(wi)
se_fixed = np.sqrt(1/np.sum(wi))
z_fixed = mu_fixed / se_fixed
p_fixed = 2*(1 - stats.norm.cdf(abs(z_fixed)))

# DerSimonian–Laird tau^2
ybar = mu_fixed
Q = np.sum(wi*(yi - ybar)**2)
df_Q = len(yi) - 1
C = np.sum(wi) - (np.sum(wi**2)/np.sum(wi))
tau2 = max(0, (Q - df_Q) / C) if C > 0 else 0.0

# Random-effects
wi_star = 1 / (vi + tau2)
mu_re = np.sum(wi_star*yi) / np.sum(wi_star)
se_re = np.sqrt(1/np.sum(wi_star))
z_re = mu_re / se_re
p_re = 2*(1 - stats.norm.cdf(abs(z_re)))
ci_re = (mu_re - 1.96*se_re, mu_re + 1.96*se_re)

# I^2
I2 = max(0.0, (Q - df_Q) / Q) * 100 if Q > 0 else 0.0

# Back-transform logit pooled effect to % scale
def inv_logit(x): return 1/(1+np.exp(-x))
pooled_prop = inv_logit(mu_re)
ci_prop = (inv_logit(ci_re[0]), inv_logit(ci_re[1]))

print("RANDOM-EFFECTS (DerSimonian–Laird)")
print(f"Pooled logit(effect) = {mu_re:.4f} ± {1.96*se_re:.4f}")
print(f"z = {z_re:.3f}, p = {p_re:.4g}")
print(f"95% CI (logit scale) = ({ci_re[0]:.4f}, {ci_re[1]:.4f})")
print(f"Heterogeneity: Q = {Q:.3f} (df={df_Q}), I² = {I2:.1f}%, tau² = {tau2:.5f}")

print("\nBack-transformed pooled effect on percentage scale:")
print(f"Pooled {PRIMARY_METRIC} = {pooled_prop*100:.2f}%  (95% CI: {ci_prop[0]*100:.2f}% – {ci_prop[1]*100:.2f}%)")

### Forest plot on logit scale with reference line at logit(0.)

In [ ]:
se = np.sqrt(vi)
# Adjust figure size dynamically based on the number of studies
fig, ax = plt.subplots(figsize=(12, max(5, 0.4 * len(meta_df) + 3)))

# Define vertical positions for studies and the overall effect
y_pos = np.arange(len(meta_df))[::-1]
overall_y = -1.5  

# 1. Plot Individual Study Error Bars and Markers
# We used marker='s' (square) and scale size by precision if desired, 
# but keeping 'o' as per our best code preference.
ax.errorbar(yi, y_pos, xerr=1.96*se, fmt='o', color='black', 
            ecolor='gray', capsize=5, elinewidth=1, label='Individual Study')

# 2. Add Pooled Effect (The "Heavy" Red Diamond)
# Using a polygon for a professional "diamond" look instead of errorbar fmt='d'
diamond_h = 0.4
diamond_x = [ci_re[0], mu_re, ci_re[1], mu_re]
diamond_y = [overall_y, overall_y + diamond_h, overall_y, overall_y - diamond_h]
ax.fill(diamond_x, diamond_y, color='darkred', zorder=5, label='Pooled Effect')

# 3. Add Vertical Reference Lines
ax.axvline(0, color='blue', linestyle='--', alpha=0.5, label='Logit Scale (0.0)')
ax.axvline(mu_re, color='darkred', linestyle=':', alpha=0.3)

# 4. Include N(Eval) and Logit F1 [95% CI] Text Columns
# We used the ax.text method to align columns precisely
for i, (idx, row) in enumerate(meta_df.iterrows()):
    y = y_pos[i]
    
    # Left Column: Study ID
    ax.text(-8.8, y, str(row['Study_ID']), ha='left', va='center', fontsize=10)
    
    # Second Column: N (Eval)
    ax.text(-6.0, y, f"{int(row['N_eval'])}", ha='center', va='center', fontsize=10)
    
    # Right Column: Numerical CI text
    ci_text = f"{yi[i]:.2f} [{yi[i]-1.96*se[i]:.2f}, {yi[i]+1.96*se[i]:.2f}]"
    ax.text(7.6, y, ci_text, ha='left', va='center', fontsize=10)

# 5. Add Overall Summary Text
ax.text(-8.5, overall_y, 'Overall (Random-Effects)', ha='left', va='center', fontweight='bold', color='darkred')
overall_ci_text = f"{mu_re:.2f} [{ci_re[0]:.2f}, {ci_re[1]:.2f}]"
ax.text(7.6, overall_y, overall_ci_text, ha='left', va='center', fontweight='bold', color='darkred')

# 6. Headers
header_y = len(meta_df) + 0.5
ax.text(-8.8, header_y, 'Study ID', ha='left', va='center', fontweight='bold')
ax.text(-6.0, header_y, 'N (Eval)', ha='center', va='center', fontweight='bold')
ax.text(0, header_y, 'Reference line', ha='center', va='center', fontweight='bold')
ax.text(7.6, header_y, 'Logit F1 [95% CI]', ha='left', va='center', fontweight='bold')
ax.axhline(header_y - 0.4, color='black', linewidth=1.5)

# 7. Final Formatting
ax.set_yticks([])
ax.set_xlabel('Effect (logit F1-score) with 95% CI', fontweight='bold')
ax.set_xlim(-9, 11)
ax.set_ylim(overall_y - 1.5, header_y + 1)

# Remove spines for a clean "table" look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# Background stripes for readability
for i in range(len(meta_df)):
    if i % 2 == 1:
        ax.axhspan(y_pos[i]-0.5, y_pos[i]+0.5, color='#f5f5f5', zorder=-1)

ax.legend(loc='lower center', bbox_to_anchor=(0.8, -0.05), ncol=3, frameon=True)
plt.tight_layout()
plt.savefig('Forest_Plot_ok22220.svg')
plt.show()

## Subgroup meta-analysis + Q_between test

    We Performed subgroup meta-analysis including Heterogeneity (I^2 and Tau^2)
    and Confidence Intervals (Lower_CI, Upper_CI).
    We Used Random-Effects (DerSimonian-Laird) for within-subgroup pooling.
    

In [ ]:
def perform_subgroup_analysis(meta_df, group_col):
    
    # Filter out missing data
    data = meta_df.dropna(subset=[group_col, '_yi', '_sei']).copy()
    data['_vi'] = data['_sei']**2  # Variance
    
    results = []
    subgroups = data[group_col].unique()
    
    print(f"\n--- Subgroup Analysis by: {group_col} ---")
    
    for sub in subgroups:
        sub_data = data[data[group_col] == sub]
        k = len(sub_data)
        if k < 2: continue 

        # 1. Fixed-Effect intermediate steps for Heterogeneity estimation
        w_fe = 1 / sub_data['_vi']
        sum_w_fe = np.sum(w_fe)
        weighted_yi_fe = np.sum(sub_data['_yi'] * w_fe) / sum_w_fe
        
        # Q-statistic for this subgroup
        q_sub = np.sum(w_fe * (sub_data['_yi'] - weighted_yi_fe)**2)
        df_sub = k - 1
        
        # 2. Estimate Tau-squared (DerSimonian-Laird method)
        # c is a scaling factor based on the weights
        c = sum_w_fe - (np.sum(w_fe**2) / sum_w_fe)
        tau2 = max(0, (q_sub - df_sub) / c)
        
        # 3. Estimate I-squared (%)
        i2 = max(0, 100 * (q_sub - df_sub) / q_sub) if q_sub > 0 else 0
        
        # 4. Random-Effects Pooling (incorporating tau2 into weights)
        w_re = 1 / (sub_data['_vi'] + tau2)
        sum_w_re = np.sum(w_re)
        pooled_yi = np.sum(sub_data['_yi'] * w_re) / sum_w_re
        pooled_se = np.sqrt(1 / sum_w_re)
        
        # 5. Calculate 95% Confidence Intervals
        lower_ci = pooled_yi - 1.96 * pooled_se
        upper_ci = pooled_yi + 1.96 * pooled_se
        
        results.append({
            'Subgroup': sub,
            'k': k,
            'Pooled_yi': pooled_yi,
            'Lower_CI': lower_ci,
            'Upper_CI': upper_ci,
            'I2 (%)': i2,
            'Tau2': tau2,
            'Q_sub': q_sub,
            'p_het': 1 - stats.chi2.cdf(q_sub, df_sub)
        })

    res_df = pd.DataFrame(results)
    
    # Q_between calculation (Moderator test)
    # Using the standard Q_total - sum(Q_within) approach
    overall_w = 1 / data['_vi']
    overall_yi = np.sum(data['_yi'] * overall_w) / np.sum(overall_w)
    q_total = np.sum(overall_w * (data['_yi'] - overall_yi)**2)
    q_within_total = res_df['Q_sub'].sum()
    
    q_between = q_total - q_within_total
    df_between = len(results) - 1
    p_between = 1 - stats.chi2.cdf(q_between, df_between) if df_between > 0 else 1

    # Displaying all critical columns
    print(res_df[['Subgroup', 'k', 'Pooled_yi', 'Lower_CI', 'Upper_CI', 'I2 (%)', 'p_het']])
    print(f"Q_between: {q_between:.4f}, df: {df_between}, p-value: {p_between:.4f}")
    
    return res_df, q_between, p_between

# List of requested subgroups
subgroup_list = [
    'Domain', 
    'Robot_Type', 
    'Imaging_Type', 
    'Autonomy_Level_img'
]

# Dictionary to store all Q-test results for our paper
final_q_tests = {}

In [ ]:
# 1. Initialize a list to hold all subgroup results for a single master table
master_results_list = []

print("Generating Final Meta-Analysis Summary Table...")

for group in subgroup_list:
    # Run the analysis function we refined
    res_df, q_b, p_b = perform_subgroup_analysis(meta_df, group)
    
    # Add a column to identify which moderator this belongs to
    res_df['Moderator'] = group
    
    # Store Q-test results for the discussion section
    final_q_tests[group] = {'Q_between': q_b, 'p_between': p_b}
    
    # Append to our master list
    master_results_list.append(res_df)

# 2. Concatenate all subgroups into one DataFrame
final_summary_table = pd.concat(master_results_list, ignore_index=True)

# 3. Format the table for readability (rounding and ordering columns)
final_summary_table = final_summary_table[[
    'Moderator', 'Subgroup', 'k', 'Pooled_yi', 'Lower_CI', 'Upper_CI', 'I2 (%)', 'Tau2', 'p_het'
]].round(3)

# 4. Final Output
print("\n" + "="*80)
print("FINAL SUBGROUP ANALYSIS SUMMARY")
print("="*80)
print(final_summary_table.to_string(index=False))
print("="*80)

# Optional: Save to CSV for Excel/LaTeX conversion
final_summary_table.to_csv("Meta_Analysis_Subgroup_Results.csv", index=False)

In [ ]:
def inv_logit(x):
    return 1 / (1 + np.exp(-x))

def subgroup_stats(df_sub):
    yi_s = df_sub['_yi'].values
    vi_s = df_sub['_vi'].values
    wi_s = 1 / vi_s

    # Fixed-effect mean (for Q)
    ybar_s = np.sum(wi_s * yi_s) / np.sum(wi_s)
    Q_s = np.sum(wi_s * (yi_s - ybar_s)**2)
    df_s = len(yi_s) - 1

    C_s = np.sum(wi_s) - (np.sum(wi_s**2) / np.sum(wi_s))
    tau2_s = max(0, (Q_s - df_s) / C_s) if C_s > 0 else 0

    wi_star_s = 1 / (vi_s + tau2_s)
    mu_s = np.sum(wi_star_s * yi_s) / np.sum(wi_star_s)
    se_s = np.sqrt(1 / np.sum(wi_star_s))

    ci_low = mu_s - 1.96 * se_s
    ci_high = mu_s + 1.96 * se_s

    I2_s = max(0.0, (Q_s - df_s) / Q_s) * 100 if Q_s > 0 else 0.0

    return {
        'mu_logit': mu_s,
        'SE': se_s,
        'CI_low_logit': ci_low,
        'CI_high_logit': ci_high,
        'mu_F1': inv_logit(mu_s),
        'CI_low_F1': inv_logit(ci_low),
        'CI_high_F1': inv_logit(ci_high),
        'tau2': tau2_s,
        'I2_%': I2_s,
        'k': len(df_sub)
    }


def meta_subgroup(meta_df, by='Domain'):
    if by not in meta_df.columns:
        raise ValueError(f"Subgroup variable '{by}' not found.")

    res = []
    Qw = 0.0  # within-group Q

    for g, dfg in meta_df.groupby(by):
        stats_s = subgroup_stats(dfg)

        # FE Q within subgroup (for Q_between)
        yi_g = dfg['_yi'].values
        vi_g = dfg['_vi'].values
        wi_g = 1 / vi_g
        ybar_g = np.sum(wi_g * yi_g) / np.sum(wi_g)

        Qw += np.sum(wi_g * (yi_g - ybar_g)**2)

        res.append({
            by: g,
            'k': stats_s['k'],
            'mu_logit': stats_s['mu_logit'],
            'CI_low_logit': stats_s['CI_low_logit'],
            'CI_high_logit': stats_s['CI_high_logit'],
            'mu_F1': stats_s['mu_F1'],
            'CI_low_F1': stats_s['CI_low_F1'],
            'CI_high_F1': stats_s['CI_high_F1'],
            'tau2': stats_s['tau2'],
            'I2_%': stats_s['I2_%']
        })

    # Q_between (Qt must already exist from main meta-analysis)
    Qt = Q
    Qb = Qt - Qw
    df_b = meta_df[by].nunique() - 1
    p_Qb = 1 - stats.chi2.cdf(Qb, df_b)

    out = pd.DataFrame(res)
    out = out.sort_values('mu_F1', ascending=False)

    return out, Qb, df_b, p_Qb

In [ ]:
for by in ['Domain', 'Robot_Type', 'Imaging_Type', 'Autonomy_Level_img']:

    if by in meta_df.columns:
        print(f"\n=== Subgroup by {by} ===")

        sub_table, Qb, dfb, pQb = meta_subgroup(meta_df, by=by)

        display(
            sub_table[[by, 'k',
                       'mu_F1', 'CI_low_F1', 'CI_high_F1',
                       'I2_%', 'tau2']]
        )

        print(f"Q_between = {Qb:.3f} (df = {dfb}), p = {pQb:.4f}")

### A 2x2 forest plot grid for the specified moderators. Including Pooled Effect, CIs, and Q-test statistics in each subplot (moderator)

In [ ]:
def plot_subgroup_forest(final_summary_table, final_q_tests):
    
    # Define the 4 moderators for the 2x2 grid
    plot_moderators = ['Domain', 'Robot_Type', 'Imaging_Type', 'Autonomy_Level_img']
    fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
    axes = axes.flatten()
    
    # Custom colors for each subplot to match the previous matrix style
    colors = ['#4A5568', '#DD6B20', '#38A169', '#3182CE']

    for i, mod in enumerate(plot_moderators):
        ax = axes[i]
        # Filter data for this specific moderator
        mod_data = final_summary_table[final_summary_table['Moderator'] == mod].copy()
        
        # Sort so the plot looks organized
        mod_data = mod_data.sort_values('Pooled_yi', ascending=True)
        
        # Plot the error bars (Confidence Intervals)
        y_pos = np.arange(len(mod_data))
        ax.errorbar(mod_data['Pooled_yi'], y_pos, 
                    xerr=[mod_data['Pooled_yi'] - mod_data['Lower_CI'], 
                          mod_data['Upper_CI'] - mod_data['Pooled_yi']],
                    fmt='d', color=colors[i], ecolor=colors[i], 
                    capsize=5, markersize=8, label='Pooled Effect (95% CI)')

        # Add text labels for the values
        for j, row in enumerate(mod_data.itertuples()):
            ax.text(row.Upper_CI + 0.05, j, f"{row.Pooled_yi:.3f} [{row.Lower_CI:.3f}, {row.Upper_CI:.3f}]", 
                    va='center', fontsize=10, fontweight='bold')

        # Formatting each subplot
        ax.set_yticks(y_pos)
        ax.set_yticklabels(mod_data['Subgroup'], fontsize=11, fontweight='bold')
        ax.set_title(f"Moderator: {mod}", fontsize=13, loc='left', fontweight='bold', color=colors[i])
        ax.axvline(x=final_summary_table['Pooled_yi'].mean(), color='red', linestyle='--', alpha=0.5) # Reference line
        
        # Add Q-test and P-value statistics inside the plot
        q_stat = final_q_tests[mod]['Q_between']
        p_val = final_q_tests[mod]['p_between']
        stats_text = f"$Q_{{between}}$: {q_stat:.3f}\n$p$: {p_val:.3f}"
        ax.text(0.05, 0.85, stats_text, transform=ax.transAxes, 
                bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'), fontsize=10)

    # Global labels
    fig.text(0.5, 0.04, 'Pooled Effect Size (logit F1-score)', ha='center', fontsize=12, fontweight='bold')
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])
    plt.savefig('subgroup analysis final results ok.png', dpi = 600)
    plt.show()

# Execute the plotting function
plot_subgroup_forest(final_summary_table, final_q_tests)

### Random-Effects Meta-Regression: R-squared, P-values, and Residual Heterogeneity (I-squared).

In [ ]:
def perform_enhanced_meta_regression(df, moderator_col):
    
    data = meta_df.dropna(subset=[moderator_col, '_yi', '_sei']).copy()
    data['_vi'] = data['_sei']**2
    
    X = sm.add_constant(data[moderator_col])
    # WLS for statistics
    model = sm.WLS(data['_yi'], X, weights=1/data['_vi']).fit()
    
    # Residual Heterogeneity calculations
    q_res = model.ssr 
    df_res = len(data) - 2
    i2_res = max(0, 100 * (q_res - df_res) / q_res) if q_res > 0 else 0
    
    r_squared = model.rsquared
    p_value = model.pvalues.iloc[1]
    slope = model.params.iloc[1]
    
    return model, data, i2_res, r_squared, p_value, slope

def plot_1x3_meta_regression(df, moderator_list):
    """
    Generates a 1x3 panel of bubble plots for three separate moderators.
    """
    fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
    colors = ['#3182CE', '#E53E3E', '#38A169'] # Blue, Red, Green
    
    for i, moderator in enumerate(moderator_list):
        # Run analysis for the specific moderator
        model, data, i2_res, r2, p_val, slope = perform_enhanced_meta_regression(df, moderator)
        ax = axes[i]
        
        # 1. Bubble size = Precision
        weights = 1 / (data['_sei']**2)
        size = (weights / weights.max()) * 500 
        
        # 2. Plot studies
        ax.scatter(data[moderator], data['_yi'], s=size, 
                   color=colors[i], alpha=0.4, edgecolors='black', label='Studies')
        
        # 3. Regression Line & 95% Confidence Interval
        x_range = np.linspace(data[moderator].min(), data[moderator].max(), 100)
        predictions = model.get_prediction(sm.add_constant(x_range))
        pred_summary = predictions.summary_frame(alpha=0.05)
        
        ax.plot(x_range, pred_summary['mean'], color='black', lw=2, label='Trend')
        ax.fill_between(x_range, pred_summary['mean_ci_lower'], pred_summary['mean_ci_upper'], 
                         color=colors[i], alpha=0.1)
        
        # 4. Statistical Annotation Box
        stats_text = (
            f"Slope (β): {slope:.4f}\n"
            f"R²: {r2:.3f}\n"
            f"p: {p_val:.4e}\n"
            f"Res. I²: {i2_res:.1f}%"
        )
        ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
                fontsize=11, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.3', 
                facecolor='white', alpha=0.8, edgecolor='gray'))

        # Formatting
        ax.set_xlabel(f'{moderator}', fontsize=12, fontweight='bold')
        if i == 0:
            ax.set_ylabel('Effect Size (Logit F1-Score)', fontsize=12, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    #plt.savefig('combined_meta_regression_1x3.png', dpi=600)
    plt.savefig('combined_meta_regression_1x3_2.svg')

    plt.show()

# Run for 3 chosen moderators
my_moderators = ['N_eval', 'Resolution_MP','Publication_Year']
plot_1x3_meta_regression(df, my_moderators)

## Correlation Matrix and VIF scores for the chosen moderators.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def check_multicollinearity(meta_df, moderator_cols):
    
    # 1. Prepare the data (only the columns of interest)
    mc_data = meta_df[moderator_cols].dropna()
    
    # 2. Generate the Correlation Matrix
    corr_matrix = mc_data.corr()
    
    # 3. Create a Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', linewidths=0.5)
    # plt.title('Moderator Correlation Matrix', fontweight='bold', pad=20)
    plt.savefig('moderator correlation matrix.png', dpi = 600, bbox_inches='tight')
    plt.show()
    
    # 4. Calculate Variance Inflation Factor (VIF)
    # VIF measures how much the variance of a regression coefficient is inflated due to collinearity
    vif_data = pd.DataFrame()
    vif_data["Moderator"] = mc_data.columns
    
    # Add constant for VIF calculation logic
    X_vif = sm.add_constant(mc_data)
    vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(1, X_vif.shape[1])]
    
    print("\n--- Variance Inflation Factor (VIF) Results ---")
    print(vif_data.to_string(index=False))
    print("\n*Interpretation: VIF < 5 is generally acceptable; VIF > 10 indicates high multicollinearity.*")
    
    return corr_matrix, vif_data

# Usage:
corr, vif = check_multicollinearity(df, ['N_eval', 'Resolution_MP','Publication_Year'])

In [ ]:
# 1. DATA ALIGNMENT (Safest way to avoid the size mismatch error)
yi_arr = np.array(meta_df['_yi']).flatten()
vi_arr = np.array(meta_df['_vi']).flatten()
se = np.sqrt(vi_arr)
precision = 1/se
std_effect = yi_arr / se

# 2. RUN EGGER'S REGRESSION
X = sm.add_constant(precision)
eg_model = sm.OLS(std_effect, X).fit()
intercept, slope = eg_model.params
se_intercept = eg_model.bse[0]
p_val = eg_model.pvalues[0]
t_stat = eg_model.tvalues[0]

# 3. CREATE COMBINED FIGURE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=600)
plt.rcParams['font.family'] = 'serif'
#rcParams['font.sans-serif'] = ['Tahoma']

# --- LEFT PLOT: FUNNEL PLOT ---
# ax1.scatter(yi_arr, se, color='#2c3e50', alpha=0.6, edgecolors='white', s=60)
ax1.scatter(yi_arr, se, color='royalblue', alpha=0.6, edgecolors='black', s=60)
# Use pooled effect from fit (mu_re)
mu_re = meta_df.get('mu_re', np.mean(yi_arr)) # Fallback to mean if mu_re missing
ax1.axvline(mu_re, color='#e74c3c', linestyle='--', lw=2, label=f'Pooled Effect')

ax1.invert_yaxis()  # Standard error 0 at the top
ax1.set_xlabel('Effect Size (logit)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Standard Error', fontsize=13, fontweight='bold')
ax1.set_title('[A]', fontsize=13, fontweight='bold', loc='center')
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(loc='upper right')

# --- RIGHT PLOT: EGGER'S REGRESSION ---
#ax2.scatter(precision, std_effect, color='#2c3e50', alpha=0.6, edgecolors='white', s=60) 
ax2.scatter(precision, std_effect, color='royalblue', alpha=0.6, edgecolors='black', s=60)
# Add Regression Line
x_range = np.linspace(0, np.max(precision) * 1.1, 100)
y_line = intercept + slope * x_range
ax2.plot(x_range, y_line, color='#e74c3c', lw=2.5, label="Regression Line")
ax2.axhline(0, color='black', linestyle='--', alpha=0.3)

# Statistics Box for Egger's
bias_status = "Significant Bias" if p_val < 0.05 else "No Significant Bias"
stats_text = (
    "Egger's Test Statistics\n"
    f"Intercept (Bias): {intercept:.3f}\n"
    f"Std. Error: {se_intercept:.3f}\n"
    f"p-value: {p_val:.4f}\n"
    f"Result: {bias_status}"
)
props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='#bdc3c7')
ax2.text(0.05, 0.95, stats_text, transform=ax2.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

ax2.set_xlabel('Precision (1/SE)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Standardized Effect (yi/SE)', fontsize=13, fontweight='bold')
ax2.set_title("[B]", fontsize=13, fontweight='bold', loc='center')
ax2.grid(True, linestyle=':', alpha=0.6)

# FINAL TOUCHES
#plt.suptitle('Publication Bias Assessment', fontsize=16, y=1.02)
plt.tight_layout()

# Save as SVG
plt.savefig('Publication_Bias.svg', bbox_inches='tight')
plt.show()

print(f"Combined figure saved successfully. Analyzed {len(yi_arr)} studies.")

In [ ]:
# Fixed‑τ² Random‑Effects Model
def random_effects_tau2(yi, vi, tau2):
    wi_star = 1 / (vi + tau2)
    mu = np.sum(wi_star * yi) / np.sum(wi_star)
    se = np.sqrt(1 / np.sum(wi_star))
    lcl = mu - 1.96 * se
    ucl = mu + 1.96 * se
    return mu, lcl, ucl

# Back‑transformation (logit → F1)
def inv_logit(x):
    return np.exp(x) / (1 + np.exp(x))

In [ ]:
# Full‑model pooled estimate (reference line)
tau2_reported = 1.30849 

mu_full, lcl_full, ucl_full = random_effects_tau2(
    meta_df['_yi'].values,
    meta_df['_vi'].values,
    tau2_reported
)

mu_full_F1 = inv_logit(mu_full)

In [ ]:
#Leave‑One‑Out (loo) computation
loo_rows = []

for i in range(len(meta_df)):
    tmp = meta_df.drop(index=meta_df.index[i]).copy()

    mu, lcl, ucl = random_effects_tau2(
        tmp['_yi'].values,
        tmp['_vi'].values,
        tau2_reported
    )

    loo_rows.append({
        'Study': meta_df.loc[meta_df.index[i], 'Study_ID'],
        'mu': mu,
        'lcl': lcl,
        'ucl': ucl,
        'mu_F1': inv_logit(mu),
        'delta_mu': mu - mu_full
    })

loo = pd.DataFrame(loo_rows)

In [ ]:
# Forest‑style LOO Plot
fig, ax = plt.subplots(figsize=(7, 12))

y = np.arange(len(loo))

# CI bars
ax.hlines(
    y=y,
    xmin=loo['lcl'],
    xmax=loo['ucl'],
    color='lightgray',
    linewidth=2
)

# Points
ax.plot(loo['mu'], y, 'D', color='tab:blue', markersize=4)

# Full pooled reference
ax.axvline(mu_full, color='red', linestyle='--', linewidth=1.3)

# Labels
ax.set_yticks(y)
ax.set_yticklabels([f"Omited {s}" for s in loo['Study']])
ax.invert_yaxis()

ax.set_xlabel("Pooled Effect Size (logit[F1])")
# ax.set_title("Leave-One-Out Sensitivity Analysis")

# Numeric overlays
for i, row in loo.iterrows():
    ax.text(
        row['ucl'] + 0.01,
        i,
        f"{row['mu']:.3f}  ({row['mu_F1']:.3f})",
        va='center',
        fontsize=8
    )

# Annotation
ax.text(
    mu_full,
    -1,
    f"overall effect size  =   {mu_full:.3f}  (F1 = {mu_full_F1:.3f})",
    ha='center',
    color='red',
    fontsize=9
)

plt.tight_layout()
plt.savefig("loo_forest_plot.svg", bbox_inches="tight")
plt.show()
